### 대분류(Section) 노드 추가

> Triple
> - (ParentCompany)-[:IN_INDUSTRY]->(s:Section)
> - (SubsidiaryCompany)-[:IN_INDUSTRY]->(s:Section)
>    
``` text
[노드 적재 결과]
{'label': 'Industry', 'count': 455}
{'label': 'ParentCompany', 'count': 565}
{'label': 'Region', 'count': 21}
{'label': 'SubsidiaryCompany', 'count': 7112}

[관계 적재 결과]
{'relation': 'AFFILIATED_WITH', 'count': 479}
{'relation': 'HAS_SUBSIDIARY', 'count': 4641}
{'relation': 'IN_INDUSTRY', 'count': 1766}
{'relation': 'LOCATED_IN', 'count': 7057}
{'relation': 'RELATED_TO_INDUSTRY', 'count': 4}
```

In [ ]:
from dotenv import load_dotenv
import pandas as pd
import os
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # 지우기 규칙 위반 에러

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것

def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]

In [ ]:
section_query = """
    CREATE (s:Section {
        {name : '금융', category: ['SPC', '펀드', '지주', '은행', '증권', '보험']},
        {name : '제조', category: ['전자', '자동차', '화학', '소재', '기계']},
        {name : 'IT·미디어', category: ['소프트웨어', '통신', '게임', '방송', '콘텐츠']},
        {name : '부동산·건설', category: ['부동산', '임대', '건설', '시공']},
        {name : '서비스', category: ['호텔', '교육', '컨설팅', '연구', '정비']},
        {name : '유통·물류', category: ['도소매', '무역', '운송', '창고']},
        {name : '바이오·헬스케어', category: ['제약', '의료', '화장품']},
        {name : '에너지·환경', category: ['발전', '태양광', '폐기물']},
        {name : '식품·농업', category: ['식품', '외식', '농축산']},
        {name : '모름', category: ['기타', '-']}
        }
    )
"""

